# OpenAI Models

**Module:** 06 — LLM Models

OpenAI model families, tool/function calling, and vision/multimodal patterns—curriculum-oriented, with placeholder APIs you can adapt as SKUs evolve.


## How to Use This Notebook

Read each section as a mini-lesson, run every code cell, then change inputs to stress-test your intuition. API examples use placeholders such as `YOUR_API_KEY` or `os.getenv(...)` — never hard-code secrets.

Each major topic includes: definition, why it matters, how it works, intuition, pitfalls, when-to-use guidance, practical demos, and a short exercise.


## Learning Objectives

By the end of this notebook, you will be able to:

- Orient across OpenAI model families used in apps
- Build correct tool/function-calling message flows
- Sketch vision/multimodal requests safely
- Choose model tiers by task complexity and cost


## Model Families (as of curriculum)

**Definition.** OpenAI exposes multiple chat/reasoning/realtime/embedding families; treat curriculum names as patterns—always verify current SKUs in docs.

**Why it matters.** Family choice drives latency, cost, tool reliability, and multimodal support.

**How it works.** Route simple tasks to small/fast models; reserve flagship/reasoning for hard jobs; keep embeddings separate.

**Intuition.** A product line: compact cars vs trucks vs specialty vehicles.

**Common pitfalls.**
- Hard-coding retired model names
- Using a flagship model for every classification
- Ignoring rate limits and token prices

**When to use.** Default hosted option for many prototypes and production apps.

| Line / SKU | Strength focus | Notes |
|------------|----------------|-------|
| Flagship chat | Hard reasoning/quality | Higher $ / latency |
| Mini / small | High QPS, extraction | Default workhorse |
| Reasoning lines | Multi-step problems | Use selectively |
| Embeddings | Retrieval vectors | Not for chat |

```mermaid
flowchart LR
  T[Task] --> R{Router}
  R -->|simple| M[Mini]
  R -->|hard| F[Flagship]
  R -->|retrieve| E[Embeddings]
```


In [ ]:
# Demo 1 — family router
def pick_model(task: str) -> str:
    return {
        "classify": "gpt-4.1-mini",
        "rag_answer": "gpt-4.1-mini",
        "hard_reason": "o-series-or-flagship",
        "embed": "text-embedding-3-large",
    }.get(task, "gpt-4.1-mini")
for t in ["classify", "hard_reason", "embed"]:
    print(t, "→", pick_model(t))


In [ ]:
# Demo — OpenAI chat request/response shape (placeholder key)
import json
YOUR_API_KEY = "YOUR_API_KEY"  # e.g. os.getenv("API_KEY")
request = {
  "model": "gpt-4.1-mini",
  "messages": [
    {"role": "system", "content": "You are a precise assistant."},
    {"role": "user", "content": "Give a 2-bullet overview."},
  ],
  "temperature": 0.3,
}
response = {
  "id": "chatcmpl_demo",
  "choices": [{"message": {"role": "assistant", "content": "- Point A\n- Point B"}, "finish_reason": "stop"}],
  "usage": {"prompt_tokens": 42, "completion_tokens": 18},
}
print("REQUEST\n", json.dumps(request, indent=2))
print("RESPONSE\n", json.dumps(response, indent=2))
print("Authorization: Bearer", YOUR_API_KEY[:8] + "...")


In [ ]:
# Demo 3 — embeddings request shape
import json
YOUR_API_KEY = "YOUR_API_KEY"
req = {"model": "text-embedding-3-small", "input": ["refund policy chunk", "user question"]}
print(json.dumps(req, indent=2))
print({"data": [{"embedding": [0.01, -0.02], "index": 0}], "usage": {"total_tokens": 12}})


In [ ]:
# Demo 4 — cost compare sketch
prices = {"mini_in": 0.4, "flagship_in": 5.0}  # $/1M tok illustrative
tokens = 2_000_000
for k, p in prices.items():
    print(k, "$", tokens/1e6*p)


### Try it yourself — Model Families (as of curriculum)

1. Map three of your workloads to small vs flagship vs embedding models.
2. Write a deprecation plan if a model id disappears next quarter.


## Function Calling / Tools

**Definition.** Models can emit structured **tool calls**; your runtime executes tools and returns **tool results** as the next messages.

**Why it matters.** This is how LLMs safely touch calendars, SQL, RAG search, and payment APIs.

**How it works.** Declare JSON schemas → model selects tool+args → you run tool → append result → model finalizes.

**Intuition.** The model writes a work order; your backend does the work.

**Common pitfalls.**
- Letting the model invent tool names
- Trusting tool args without validation
- Missing the tool-result turn in the transcript

**When to use.** Any agentic or grounded app that must take actions.

```mermaid
sequenceDiagram
  participant U as User
  participant M as Model
  participant R as Runtime
  U->>M: messages
  M->>R: tool_call
  R->>M: tool result
  M->>U: final answer
```


In [ ]:
# Demo 1 — tool schema
import json
tools = [{
  "type": "function",
  "function": {
    "name": "search_kb",
    "description": "Hybrid search over policy docs",
    "parameters": {
      "type": "object",
      "properties": {"query": {"type": "string"}, "k": {"type": "integer"}},
      "required": ["query"],
    },
  },
}]
print(json.dumps(tools, indent=2))


In [ ]:
# Demo 2 — tool call message + your execution
assistant_msg = {
  "role": "assistant",
  "tool_calls": [{
    "id": "call_1",
    "type": "function",
    "function": {"name": "search_kb", "arguments": '{"query":"refund window","k":4}'},
  }],
}
def search_kb(query, k=4):
    return [{"id": "C1", "text": "Refunds within 60 days"}][:k]

import json
args = json.loads(assistant_msg["tool_calls"][0]["function"]["arguments"])
result = search_kb(**args)
tool_msg = {"role": "tool", "tool_call_id": "call_1", "content": json.dumps(result)}
print(tool_msg)


In [ ]:
# Demo 3 — validate args before side effects
def validate_search_args(args: dict):
    assert isinstance(args.get("query"), str) and args["query"].strip()
    k = int(args.get("k", 4))
    assert 1 <= k <= 20
    return {"query": args["query"].strip(), "k": k}
print(validate_search_args({"query": " shipping ", "k": 3}))


### Try it yourself — Function Calling / Tools

1. Design one tool schema for 'create_ticket' with required fields.
2. Show the four message roles in a successful tool round-trip.


## Vision & Multimodal

**Definition.** Multimodal models accept images (and sometimes audio) alongside text in the messages array.

**Why it matters.** Unlocks screenshot QA, document photos, UI debugging, and multimodal RAG.

**How it works.** Send image URLs or base64 parts with text instructions; still apply safety and size limits.

**Intuition.** Show-and-tell for models.

**Common pitfalls.**
- Shipping raw PII images to third parties without review
- Assuming OCR is perfect
- Huge images blowing token/byte budgets

**When to use.** When the answer depends on pixels, not only text.


In [ ]:
# Demo 1 — multimodal message shape
import json
msg = {
  "role": "user",
  "content": [
    {"type": "text", "text": "Extract the total from this receipt."},
    {"type": "image_url", "image_url": {"url": "https://example.com/receipt.png"}},
  ],
}
print(json.dumps(msg, indent=2))


In [ ]:
# Demo 2 — base64 placeholder (do not embed secrets)
import base64
fake_png_bytes = b"\x89PNG_DEMO"
b64 = base64.b64encode(fake_png_bytes).decode()
print("data:image/png;base64," + b64[:20] + "...")


In [ ]:
# Demo 3 — guardrails
def allow_image_upload(user_role, contains_sensitive):
    if contains_sensitive and user_role != "approved_agent":
        return False
    return True
print(allow_image_upload("customer", True), allow_image_upload("approved_agent", True))


### Try it yourself — Vision & Multimodal

1. Write a prompt that forces structured JSON extraction from an image.
2. List privacy checks before enabling screenshot uploads.


## Glossary

- **tool call**: Structured function invocation from the model
- **multimodal part**: Non-text content piece in a message


### Workshop drill — OpenAI Models (1)

Fill a comparison row: strengths, limits, typical SKU/API name, and one eval signal.


In [ ]:
# Workshop drill 1 — OpenAI Models
import json
print(json.dumps({
  'family': 'OpenAI Models',
  'strengths': ['...'],
  'limits': ['...'],
  'api_name': '...',
  'eval_signal': '...'
}, indent=2))


### Workshop drill — OpenAI Models (2)

Draft a realistic request JSON with YOUR_API_KEY placeholder (do not call the network).


In [ ]:
# Workshop drill 2 — OpenAI Models
import json
YOUR_API_KEY='YOUR_API_KEY'
req={'model':'MODEL_ID','messages':[{'role':'user','content':'Hello'}]}
print(json.dumps(req, indent=2))
print('Authorization: Bearer', YOUR_API_KEY[:8]+'...')


### Workshop drill — OpenAI Models (3)

Write a go/no-go for this family on a regulated enterprise FAQ bot.


In [ ]:
# Workshop drill 3 — OpenAI Models
checks=['license','data_residency','tool_calling','vision_needed','cost']
for c in checks:
    print(f'[ ] {c}: pass/fail because ...')


### Workshop drill — OpenAI Models (4)

List three prompts that stress this family's claimed strengths.


In [ ]:
# Workshop drill 4 — OpenAI Models
for i,p in enumerate(['prompt1','prompt2','prompt3'],1):
    print(i, p)


### Workshop drill — OpenAI Models (5)

Cost sketch: estimate monthly $ for 10M input + 2M output tokens.


In [ ]:
# Workshop drill 5 — OpenAI Models
in_tok, out_tok = 10_000_000, 2_000_000
in_price = out_price = 1.0  # $/1M tok placeholders — replace with card prices
print('approx $', in_tok/1e6*in_price + out_tok/1e6*out_price)


### Workshop drill — OpenAI Models (6)

Router rule: when would you escalate from a small model in this family to a larger one?


In [ ]:
# Workshop drill 6 — OpenAI Models
def route(complexity: float) -> str:
    return 'large' if complexity > 0.7 else 'small'
for c in [0.2, 0.7, 0.95]:
    print(c, route(c))


## Summary & Key Takeaways

- Route by task to mini vs flagship vs embeddings
- Tool calling is a message protocol plus your validated runtime
- Multimodal inputs need privacy and size discipline

### Practice

Implement a fake tool round-trip for search_kb end-to-end in pure Python.


## Self-Check

1. Can you explain the main idea of each section in one sentence?
2. Which technique would you use first in production, and why?
3. What failure mode should you monitor after shipping?
4. What metric would tell you the system got worse?


In [ ]:
checklist = [
    "I can restate the learning objectives",
    "I ran/adapted at least two code examples",
    "I know which env vars/keys this topic needs",
    "I noted one risk (cost, safety, latency, or quality)",
    "I can name one pitfall and its mitigation",
]
for i, item in enumerate(checklist, 1):
    print(f"{i}. [ ] {item}")
